# 🔬 Physics-Informed Deep Inverse Reconstruction for Lensless Phase-Mask Imaging

## A Step-by-Step Tutorial

**Welcome!** This notebook walks you through the entire pipeline for lensless computational imaging — from the physics of how a DiffuserCam works, to classical reconstruction algorithms, to cutting-edge deep learning approaches.

### What You'll Learn:
1. **Part 1:** What is a PSF and why does it matter?
2. **Part 2:** Simulating a lensless camera (the forward model)
3. **Part 3:** Classical reconstruction — Wiener deconvolution & ADMM
4. **Part 4:** Building an unrolled deep neural network
5. **Part 5:** Physics-guided loss functions (TV + perceptual)
6. **Part 6:** Putting it all together — full comparison

### Prerequisites:
- Basic Python & NumPy
- Some familiarity with PyTorch (or willingness to learn!)
- High school math (we'll explain the rest)

---

## Setup

First, let's import everything we need and check our setup.

In [ ]:
# Standard imports
import sys
import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
import time

# Make plots look nice
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['image.cmap'] = 'gray'

# Add project root to path
project_root = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
sys.path.insert(0, project_root)

# Import our modules
from src.forward_model import PSFGenerator, DiffuserCamForwardModel, create_test_scene
from src.classical_recon import WienerDeconvolution, ADMMReconstructor, TVDenoiser
from src.deep_recon import UnrolledADMMNetwork, SimpleUNet, CNNDenoiser
from src.losses import (
    CombinedLoss, TotalVariationLoss, DataFidelityLoss,
    compute_psnr, compute_ssim
)
from src.utils import tensor_to_numpy, numpy_to_tensor

# Check setup
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\n✅ All imports successful!")

---
# Part 1: Understanding Point Spread Functions (PSFs)

## What is a PSF?

A **Point Spread Function (PSF)** describes how a single point of light gets "spread" by an optical system.

- In a **perfect lens**: the PSF is a tiny dot (all light focused to one point)
- In a **blurry lens**: the PSF is a blob (light spreads out)
- In a **DiffuserCam**: the PSF is a complex caustic pattern (light gets randomly scattered)

The PSF completely characterizes the optical system. If we know the PSF, we can (in theory) undo its effects!

Let's generate and visualize different types of PSFs.

In [ ]:
# Create PSF generators with different types
SIZE = 256  # Image size (you can change this)
psf_gen = PSFGenerator(size=SIZE, seed=42)

# Generate three types of PSFs
psf_gaussian = psf_gen.gaussian_mixture(num_components=50)
psf_phase = psf_gen.random_phase_mask(feature_size=8.0)
psf_caustic = psf_gen.caustic_pattern(num_features=200)

# Visualize them
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, psf, title in zip(axes, 
    [psf_gaussian, psf_phase, psf_caustic],
    ['Gaussian Mixture PSF', 'Random Phase Mask PSF', 'Caustic Pattern PSF']):
    ax.imshow(psf, cmap='hot')
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.suptitle('Three Types of PSFs for Lensless Imaging', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"All PSFs sum to 1.0 (energy conservation):")
print(f"  Gaussian: {psf_gaussian.sum():.4f}")
print(f"  Phase:    {psf_phase.sum():.4f}")
print(f"  Caustic:  {psf_caustic.sum():.4f}")

### 🔍 Understanding the PSF in Fourier Domain

The Fourier transform of the PSF tells us which spatial frequencies the camera can "see".
- Bright regions in the Fourier domain → those frequencies are well-preserved
- Dark regions → those frequencies are lost (hard to reconstruct!)

A good diffuser PSF should have broad Fourier coverage (no dead zones).

In [ ]:
# Analyze the frequency content of each PSF
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for col, (psf, title) in enumerate(zip(
    [psf_gaussian, psf_phase, psf_caustic],
    ['Gaussian Mixture', 'Random Phase Mask', 'Caustic Pattern'])):
    
    # Spatial domain
    axes[0, col].imshow(psf, cmap='hot')
    axes[0, col].set_title(f'{title}\n(Spatial Domain)', fontsize=11)
    axes[0, col].axis('off')
    
    # Fourier domain (MTF - Modulation Transfer Function)
    mtf = np.abs(np.fft.fftshift(np.fft.fft2(psf)))
    axes[1, col].imshow(np.log10(mtf + 1e-10), cmap='viridis')
    axes[1, col].set_title(f'{title}\n(Fourier Domain / MTF, log scale)', fontsize=11)
    axes[1, col].axis('off')

plt.suptitle('PSF Analysis: Spatial vs. Frequency Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Notice: The Phase Mask PSF has the most uniform Fourier coverage.")
print("   This means it preserves the most information — ideal for reconstruction!")

---
# Part 2: The Forward Model — Simulating a DiffuserCam

## How Does a DiffuserCam Capture Images?

The measurement process is surprisingly simple mathematically:

$$y = \text{PSF} * x + n$$

Where:
- $x$ = the scene (what we want to capture)
- $*$ = convolution (the PSF "smears" each point of the scene)
- $n$ = sensor noise
- $y$ = the raw sensor measurement

Let's simulate this!

In [ ]:
# Choose our PSF (the random phase mask is most realistic)
psf = psf_phase

# Create different test scenes
scenes = {}
for scene_type in ['checkerboard', 'resolution_target', 'natural']:
    scenes[scene_type] = create_test_scene(size=SIZE, scene_type=scene_type)

# Visualize the test scenes
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, scene) in zip(axes, scenes.items()):
    ax.imshow(scene, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'{name.replace("_", " ").title()} Scene', fontsize=13)
    ax.axis('off')

plt.suptitle('Test Scenes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Build the forward model
noise_std = 0.01  # Try changing this! (0.001 = clean, 0.1 = very noisy)
forward_model = DiffuserCamForwardModel(psf, noise_std=noise_std).to(device)

# Simulate measurements for each scene
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

measurements = {}
for col, (name, scene) in enumerate(scenes.items()):
    # Convert to tensor
    scene_tensor = torch.from_numpy(scene).float().unsqueeze(0).unsqueeze(0).to(device)
    
    # Apply forward model
    measurement = forward_model(scene_tensor)
    measurements[name] = measurement
    
    # Display
    axes[0, col].imshow(scene, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'Original: {name}', fontsize=11)
    axes[0, col].axis('off')
    
    meas_np = measurement.squeeze().cpu().numpy()
    axes[1, col].imshow(meas_np, cmap='gray')
    axes[1, col].set_title(f'Measurement (noise={noise_std})', fontsize=11)
    axes[1, col].axis('off')

plt.suptitle('Forward Model: Scene → Measurement\n(y = PSF * x + noise)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 The measurements look like random blobs — nothing like the original scenes!")
print("   But all the information is still there, encoded in the caustic pattern.")
print("   Our job: DECODE it.")

### 🧪 Experiment: Effect of Noise Level

Let's see how noise affects the measurement and (later) reconstruction quality.

In [ ]:
# Compare different noise levels
scene = scenes['natural']
scene_tensor = torch.from_numpy(scene).float().unsqueeze(0).unsqueeze(0).to(device)

noise_levels = [0.0, 0.001, 0.01, 0.05, 0.1]
fig, axes = plt.subplots(1, len(noise_levels), figsize=(20, 4))

for ax, noise in zip(axes, noise_levels):
    model_noisy = DiffuserCamForwardModel(psf, noise_std=noise).to(device)
    meas = model_noisy(scene_tensor)
    ax.imshow(meas.squeeze().cpu().numpy(), cmap='gray')
    ax.set_title(f'Noise σ = {noise}', fontsize=11)
    ax.axis('off')

plt.suptitle('Effect of Noise on Measurements', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 3: Classical Reconstruction

Now the fun part — recovering the original image from the scrambled measurement!

## 3.1 Wiener Deconvolution

The simplest approach. In Fourier domain:

$$\hat{X}(f) = \frac{H^*(f)}{|H(f)|^2 + \lambda} \cdot Y(f)$$

Where:
- $H^*(f)$ = conjugate of the PSF's Fourier transform
- $\lambda$ = regularization parameter
- $Y(f)$ = Fourier transform of the measurement

**Pros:** Super fast (one FFT pair!)  
**Cons:** Limited quality, can't enforce non-negativity

In [ ]:
# --- Wiener Deconvolution ---

# Use the natural scene
scene = scenes['natural']
scene_tensor = torch.from_numpy(scene).float().unsqueeze(0).unsqueeze(0).to(device)

# Create forward model and measurement
forward_model = DiffuserCamForwardModel(psf, noise_std=0.01).to(device)
measurement = forward_model(scene_tensor)

# Try different regularization values
reg_values = [0.0001, 0.001, 0.01, 0.1]

fig, axes = plt.subplots(1, len(reg_values) + 2, figsize=(24, 4))

# Show original and measurement
axes[0].imshow(scene, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

axes[1].imshow(measurement.squeeze().cpu().numpy(), cmap='gray')
axes[1].set_title('Measurement', fontsize=11)
axes[1].axis('off')

for i, reg in enumerate(reg_values):
    wiener = WienerDeconvolution(forward_model.psf_fft, regularization=reg)
    recon = wiener.reconstruct(measurement)
    
    psnr = compute_psnr(recon, scene_tensor)
    
    axes[i + 2].imshow(recon.squeeze().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[i + 2].set_title(f'λ={reg}\nPSNR={psnr:.1f}dB', fontsize=11)
    axes[i + 2].axis('off')

plt.suptitle('Wiener Deconvolution: Effect of Regularization Parameter λ', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Too small λ → noisy (amplifies noise at weak frequencies)")
print("   Too large λ → blurry (over-regularized, kills high frequencies)")
print("   Sweet spot → best trade-off between noise and blur")

## 3.2 ADMM Reconstruction with Total Variation

ADMM is much more powerful than Wiener because it can incorporate **priors** — knowledge about what natural images look like.

We use **Total Variation (TV)** as our prior, which says: "natural images are mostly smooth with sharp edges."

ADMM solves:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - y\|^2 + \lambda \cdot TV(x)$$

By alternating three steps:
1. **x-update:** Solve data fidelity (easy in Fourier domain)
2. **z-update:** Apply TV denoising (proximal operator)
3. **u-update:** Update dual variable (enforce consistency)

In [ ]:
# --- ADMM Reconstruction ---

print("Running ADMM reconstruction (this may take a moment)...")
print()

admm = ADMMReconstructor(
    psf_fft=forward_model.psf_fft,
    psf_fft_conj=forward_model.psf_fft_conj,
    psf_fft_abs_sq=forward_model.psf_fft_abs_sq,
    rho=1.0,              # ADMM penalty (try: 0.1 to 10)
    lambda_tv=0.005,      # TV weight (try: 0.001 to 0.05)
    num_iters=50,         # More iterations = better (but slower)
    tv_inner_iters=15,    # Inner iterations for TV proximal operator
    verbose=True
)

admm_recon, history = admm.reconstruct(measurement, ground_truth=scene_tensor)

# Also compute Wiener for comparison
wiener = WienerDeconvolution(forward_model.psf_fft, regularization=0.001)
wiener_recon = wiener.reconstruct(measurement)

In [ ]:
# Compare Wiener vs ADMM
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

images_to_show = [
    (scene, 'Ground Truth'),
    (measurement.squeeze().cpu().numpy(), 'Measurement'),
    (wiener_recon.squeeze().cpu().numpy(), 
     f'Wiener\nPSNR={compute_psnr(wiener_recon, scene_tensor):.1f}dB'),
    (admm_recon.squeeze().cpu().numpy(), 
     f'ADMM (50 iters)\nPSNR={compute_psnr(admm_recon, scene_tensor):.1f}dB'),
]

for ax, (img, title) in zip(axes, images_to_show):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Classical Reconstruction Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot ADMM convergence
if any('psnr' in h for h in history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    iters = [h['iteration'] for h in history]
    
    # PSNR over iterations
    psnrs = [h.get('psnr', 0) for h in history]
    ax1.plot(iters, psnrs, 'b-o', markersize=3)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('PSNR (dB)')
    ax1.set_title('PSNR vs. Iteration')
    ax1.grid(True, alpha=0.3)
    
    # Primal residual (convergence indicator)
    residuals = [h['primal_residual'] for h in history]
    ax2.semilogy(iters, residuals, 'r-o', markersize=3)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Primal Residual')
    ax2.set_title('Convergence (Primal Residual)')
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle('ADMM Convergence Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"💡 PSNR improves rapidly in early iterations, then plateaus.")
    print(f"   Final PSNR: {psnrs[-1]:.2f} dB")

### 🧪 Experiment: ADMM Hyperparameter Sweep

Let's see how the TV weight λ affects ADMM reconstruction.

In [ ]:
# Sweep over TV regularization strengths
lambda_values = [0.0005, 0.002, 0.005, 0.02, 0.05]

fig, axes = plt.subplots(1, len(lambda_values), figsize=(20, 4))

for ax, lam in zip(axes, lambda_values):
    admm_sweep = ADMMReconstructor(
        forward_model.psf_fft, forward_model.psf_fft_conj, 
        forward_model.psf_fft_abs_sq,
        rho=1.0, lambda_tv=lam, num_iters=30, verbose=False
    )
    recon, _ = admm_sweep.reconstruct(measurement, ground_truth=scene_tensor)
    psnr = compute_psnr(recon, scene_tensor)
    
    ax.imshow(recon.squeeze().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'λ_TV={lam}\nPSNR={psnr:.1f}dB', fontsize=11)
    ax.axis('off')

plt.suptitle('ADMM: Effect of TV Regularization Weight', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Low λ_TV → preserves detail but keeps noise")
print("   High λ_TV → smooth/cartoon-like but removes noise")

---
# Part 4: Deep Learning — Unrolled ADMM Network

## The Key Insight: Algorithm Unrolling

What if we could **learn** the optimal parameters for ADMM instead of hand-tuning them?

**Algorithm unrolling** takes each iteration of ADMM and turns it into a **layer** of a neural network:

| ADMM Iteration | → | Network Layer |
|---|---|---|
| Fixed ρ | → | **Learnable** ρ_k (different per stage!) |
| TV proximal operator | → | **CNN denoiser** (much more expressive!) |
| Fixed step size | → | **Learnable** step size |
| 100+ iterations | → | **10 stages** (much faster!) |

The entire pipeline is differentiable, so we can train it end-to-end!

In [ ]:
# Build the unrolled ADMM network
# We'll use a smaller size for fast training in this tutorial
TRAIN_SIZE = 64  # Smaller for tutorial speed (use 128-256 for real experiments)

# Generate PSF at training size
psf_gen_train = PSFGenerator(size=TRAIN_SIZE, seed=42)
psf_train = psf_gen_train.random_phase_mask(feature_size=3.0)

# Create the network
net = UnrolledADMMNetwork(
    psf=psf_train,
    num_stages=5,         # 5 unrolled ADMM stages (try 8-10 for better quality)
    in_channels=1,        # Grayscale
    num_features=32,      # Feature channels (32 for tutorial, 64 for production)
    num_blocks_per_stage=2  # ResBlocks per denoiser (2 for tutorial, 3-4 for production)
).to(device)

# Print architecture summary
net.print_architecture()

### Training the Network

We train on synthetic data: generate random scenes, apply the forward model, and teach the network to reconstruct.

In [ ]:
# Create synthetic training data
from src.train import SyntheticLenslessDataset

train_dataset = SyntheticLenslessDataset(
    psf=psf_train,
    num_samples=100,      # Small for tutorial (use 500+ for real training)
    image_size=TRAIN_SIZE,
    noise_std=0.01
)

val_dataset = SyntheticLenslessDataset(
    psf=psf_train,
    num_samples=20,
    image_size=TRAIN_SIZE,
    noise_std=0.01
)

# Visualize some training samples
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i in range(5):
    meas, gt = train_dataset[i]
    axes[0, i].imshow(gt.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Ground Truth {i}', fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(meas.squeeze().numpy(), cmap='gray')
    axes[1, i].set_title(f'Measurement {i}', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Training Data Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Training loop (simplified for tutorial)
from torch.utils.data import DataLoader

# Hyperparameters
NUM_EPOCHS = 15        # Increase to 50+ for better results
BATCH_SIZE = 4
LEARNING_RATE = 1e-4

# Setup
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Loss: MSE + TV
tv_loss_fn = TotalVariationLoss(mode='isotropic')
lambda_tv = 0.01

# Training history
train_losses = []
val_psnrs = []

print("Starting training...")
print(f"{'Epoch':>5} | {'Train Loss':>12} | {'Val PSNR':>10} | {'Time':>8}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training ---
    net.train()
    epoch_loss = 0.0
    t0 = time.time()
    
    for measurements, targets in train_loader:
        measurements = measurements.to(device)
        targets = targets.to(device)
        
        # Forward pass
        predictions = net(measurements)
        
        # Loss = MSE + λ·TV
        mse_loss = F.mse_loss(predictions, targets)
        tv = tv_loss_fn(predictions)
        loss = mse_loss + lambda_tv * tv
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step()
    
    # --- Validation ---
    net.eval()
    total_psnr = 0.0
    n_val = 0
    with torch.no_grad():
        for measurements, targets in val_loader:
            measurements = measurements.to(device)
            targets = targets.to(device)
            predictions = net(measurements)
            total_psnr += compute_psnr(predictions, targets)
            n_val += 1
    
    avg_psnr = total_psnr / max(n_val, 1)
    val_psnrs.append(avg_psnr)
    
    elapsed = time.time() - t0
    print(f"{epoch:5d} | {avg_loss:12.6f} | {avg_psnr:8.2f} dB | {elapsed:6.1f}s")

print("\n✅ Training complete!")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, 'b-')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(val_psnrs, 'g-')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation PSNR (dB)')
ax2.set_title('Validation PSNR')
ax2.grid(True, alpha=0.3)

plt.suptitle('Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned parameters
print("Learned ADMM Parameters:")
print(f"{'Stage':>5} | {'ρ (penalty)':>12} | {'Dual Step':>10}")
print("-" * 35)

for i, stage in enumerate(net.stages):
    rho = stage.rho.item()
    step = stage.dual_step.item()
    print(f"{i:5d} | {rho:12.4f} | {step:10.4f}")

print("\n💡 Notice how the network learned different ρ values for each stage!")
print("   Early stages often have smaller ρ (exploratory), later stages have larger ρ (refined).")

---
# Part 5: Loss Function Engineering

Let's explore how different loss functions affect reconstruction quality.

## 5.1 Total Variation Loss

TV promotes piecewise-smooth images (sharp edges, flat regions).

In [ ]:
# Demonstrate TV denoising
# Add noise to a clean image and denoise with TV

clean = create_test_scene(size=TRAIN_SIZE, scene_type='natural')
clean_tensor = torch.from_numpy(clean).float().unsqueeze(0).unsqueeze(0).to(device)

# Add noise
noisy = clean_tensor + 0.1 * torch.randn_like(clean_tensor)
noisy = noisy.clamp(0, 1)

# Apply TV denoising with different strengths
tv_lambdas = [0.01, 0.05, 0.1, 0.3]

fig, axes = plt.subplots(1, len(tv_lambdas) + 2, figsize=(22, 4))

axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Clean', fontsize=11)
axes[0].axis('off')

axes[1].imshow(noisy.squeeze().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Noisy\nPSNR={compute_psnr(noisy, clean_tensor):.1f}dB', fontsize=11)
axes[1].axis('off')

for i, lam in enumerate(tv_lambdas):
    denoised = TVDenoiser.prox_tv(noisy, lambda_tv=lam, num_iters=50)
    psnr = compute_psnr(denoised, clean_tensor)
    
    axes[i + 2].imshow(denoised.squeeze().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[i + 2].set_title(f'TV λ={lam}\nPSNR={psnr:.1f}dB', fontsize=11)
    axes[i + 2].axis('off')

plt.suptitle('Total Variation Denoising', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.2 Computing Quality Metrics

Let's understand **PSNR** and **SSIM** — the two most common image quality metrics.

In [ ]:
# Compare metrics for different degradation types
clean = create_test_scene(size=64, scene_type='natural')
clean_t = torch.from_numpy(clean).float().unsqueeze(0).unsqueeze(0)

degradations = {
    'Clean': clean_t.clone(),
    'Small noise (σ=0.05)': (clean_t + 0.05 * torch.randn_like(clean_t)).clamp(0, 1),
    'Large noise (σ=0.2)': (clean_t + 0.2 * torch.randn_like(clean_t)).clamp(0, 1),
    'Blurred': F.avg_pool2d(F.pad(clean_t, (2,2,2,2), mode='reflect'), 5, stride=1),
    'Shifted (2px)': torch.roll(clean_t, shifts=2, dims=3),
}

print(f"{'Degradation':<25} | {'PSNR (dB)':>10} | {'SSIM':>8}")
print("-" * 50)

for name, degraded in degradations.items():
    # Ensure same size
    degraded = degraded[:, :, :clean_t.shape[2], :clean_t.shape[3]]
    psnr = compute_psnr(degraded, clean_t)
    ssim = compute_ssim(degraded, clean_t)
    print(f"{name:<25} | {psnr:>8.2f}dB | {ssim:>8.4f}")

print("\n💡 Notice: A shifted image has LOW PSNR but looks identical to humans.")
print("   This is why perceptual metrics (SSIM, perceptual loss) matter!")

---
# Part 6: Full Pipeline Comparison

Let's put it all together and compare all reconstruction methods side by side.

In [ ]:
# Full comparison on the training-size images
print("="*60)
print("  FULL PIPELINE COMPARISON")
print("="*60)

# Create forward model at training size
fwd_train = DiffuserCamForwardModel(psf_train, noise_std=0.01).to(device)

# Generate a test scene
test_scene = create_test_scene(size=TRAIN_SIZE, scene_type='natural')
test_tensor = torch.from_numpy(test_scene).float().unsqueeze(0).unsqueeze(0).to(device)
test_measurement = fwd_train(test_tensor)

# Method 1: Wiener
t0 = time.time()
wiener = WienerDeconvolution(fwd_train.psf_fft, regularization=0.001)
wiener_result = wiener.reconstruct(test_measurement)
wiener_time = time.time() - t0
wiener_psnr = compute_psnr(wiener_result, test_tensor)
wiener_ssim = compute_ssim(wiener_result, test_tensor)

# Method 2: ADMM
t0 = time.time()
admm_small = ADMMReconstructor(
    fwd_train.psf_fft, fwd_train.psf_fft_conj, fwd_train.psf_fft_abs_sq,
    rho=1.0, lambda_tv=0.005, num_iters=50, verbose=False
)
admm_result, _ = admm_small.reconstruct(test_measurement)
admm_time = time.time() - t0
admm_psnr = compute_psnr(admm_result, test_tensor)
admm_ssim = compute_ssim(admm_result, test_tensor)

# Method 3: Learned (unrolled ADMM network)
net.eval()
t0 = time.time()
with torch.no_grad():
    learned_result = net(test_measurement)
learned_time = time.time() - t0
learned_psnr = compute_psnr(learned_result, test_tensor)
learned_ssim = compute_ssim(learned_result, test_tensor)

# Print results table
print(f"\n{'Method':<20} | {'PSNR (dB)':>10} | {'SSIM':>8} | {'Time (s)':>10}")
print("-" * 58)
print(f"{'Wiener':<20} | {wiener_psnr:>8.2f}dB | {wiener_ssim:>8.4f} | {wiener_time:>8.4f}s")
print(f"{'ADMM (50 iters)':<20} | {admm_psnr:>8.2f}dB | {admm_ssim:>8.4f} | {admm_time:>8.4f}s")
print(f"{'Learned (5 stages)':<20} | {learned_psnr:>8.2f}dB | {learned_ssim:>8.4f} | {learned_time:>8.4f}s")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

images_final = [
    (test_scene, 'Ground Truth'),
    (test_measurement.squeeze().cpu().numpy(), 'Measurement'),
    (wiener_result.squeeze().cpu().numpy(), f'Wiener\n{wiener_psnr:.1f}dB / {wiener_ssim:.3f}'),
    (admm_result.squeeze().cpu().numpy(), f'ADMM\n{admm_psnr:.1f}dB / {admm_ssim:.3f}'),
    (learned_result.squeeze().cpu().numpy(), f'Learned\n{learned_psnr:.1f}dB / {learned_ssim:.3f}'),
]

for ax, (img, title) in zip(axes, images_final):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Reconstruction Method Comparison (PSNR / SSIM)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize intermediate stages of the unrolled network
net.eval()
with torch.no_grad():
    _, intermediates = net(test_measurement, return_intermediates=True)

fig, axes = plt.subplots(1, len(intermediates) + 1, figsize=(22, 4))

axes[0].imshow(test_measurement.squeeze().cpu().numpy(), cmap='gray')
axes[0].set_title('Input\n(Measurement)', fontsize=10)
axes[0].axis('off')

for i, intermediate in enumerate(intermediates):
    psnr = compute_psnr(intermediate.to(device), test_tensor)
    axes[i + 1].imshow(intermediate.squeeze().cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[i + 1].set_title(f'Stage {i+1}\nPSNR={psnr:.1f}dB', fontsize=10)
    axes[i + 1].axis('off')

plt.suptitle('Unrolled Network: Reconstruction Evolving Through Stages', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Watch how the image gets progressively cleaner through the stages!")
print("   Each stage refines the output of the previous one.")

---
# 🎓 Summary & Next Steps

## What We Covered

| Topic | Key Takeaway |
|-------|-------------|
| **PSFs** | A PSF characterizes how a lensless camera scrambles light |
| **Forward Model** | y = PSF * x + noise (convolution → FFT multiplication) |
| **Wiener Filter** | Fast but limited; closed-form Fourier-domain solution |
| **ADMM** | Powerful iterative algorithm with TV prior; handles constraints |
| **Unrolled Network** | ADMM iterations → neural network layers; learns optimal parameters |
| **Loss Engineering** | MSE + TV + Perceptual → balanced image quality |

## Ideas for Further Exploration

1. **Try real PSF data** from the Waller Lab DiffuserCam
2. **3D reconstruction** — reconstruct depth from a single 2D measurement
3. **Color images** — extend to RGB (3-channel) reconstruction
4. **More training data** — use natural image datasets (DIV2K, BSD500)
5. **Perceptual loss** — enable VGG perceptual loss for better visual quality
6. **Larger networks** — increase stages, features, and training epochs
7. **Noise robustness** — train with varying noise levels
8. **Comparison with U-Net** — pure data-driven vs. physics-informed

## References

- Antipa et al., "DiffuserCam: Lensless Single-exposure 3D Imaging" (Optica, 2018)
- Monakhova et al., "Learned Reconstructions for Practical Mask-Based Lensless Imaging" (2019)
- Monga et al., "Algorithm Unrolling" (IEEE SPM, 2021)
- Boyd et al., "Distributed Optimization and Statistical Learning via ADMM" (2011)

---
*Built with curiosity and PyTorch 🔥*